# NFL Data Analysis Example

This notebook shows how to connect to the NFL database and perform basic analysis.

In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [7]:
# Connect to the NFL database
db_path = Path('../data/nfl.duckdb')
conn = duckdb.connect(str(db_path))

print(f"Connected to database: {db_path}")
print("Database connection established successfully!")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

Connected to database: ../data/nfl.duckdb
Database connection established successfully!


In [ ]:
q = """
with last_plays as (
select game_id, 
        max(play_id)
        
         as last_play_id from pbp where season=2025 group by all
)
select
    pbp.game_id,
    pbp.play_id,
    pbp.home_team,
    pbp.home_score,
    pbp.away_team,
    pbp.away_score,
    case when pbp.home_score > pbp.away_score then pbp.home_team 
        when pbp.away_score > pbp.home_score then pbp.away_team
        else 'tie'
     end as winner,
 
from last_plays lp
left join pbp on lp.game_id=pbp.game_id and lp.last_play_id=pbp.play_id

"""

In [21]:
df=pd.read_sql_query(q,conn)

/var/folders/kp/8st1tbqj0px8gr_28_46kn4m0000gn/T/ipykernel_96086/3007800487.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query(q,conn)


In [24]:
df

,game_id,play_id,home_team,home_score,away_team,away_score,winner
0,2025_01_ARI_NO,4714.0,NO,13,ARI,20,ARI
1,2025_01_BAL_BUF,4693.0,BUF,41,BAL,40,BUF
2,2025_01_CAR_JAX,4245.0,JAX,26,CAR,10,JAX
3,2025_01_CIN_CLE,4205.0,CLE,16,CIN,17,CIN
4,2025_01_DAL_PHI,3968.0,PHI,24,DAL,20,PHI
5,2025_01_DET_GB,3899.0,GB,27,DET,13,GB
6,2025_01_HOU_LA,4228.0,LA,14,HOU,9,LA
7,2025_01_KC_LAC,4127.0,LAC,27,KC,21,LAC
8,2025_01_LV_NE,4532.0,NE,13,LV,20,LV
9,2025_01_MIA_IND,3683.0,IND,33,MIA,8,IND


In [45]:
df['abs_wpa']=df['wpa'].abs()

br = df[df['home_team']=='BUF'].sort_values(by='abs_wpa',ascending=False)

In [48]:
top_plays = br[['play_id','posteam','qtr','time','down','ydstogo','play_type','wpa','home_wp','abs_wpa','desc']].head(20)#.sort_values(by='play_id')

top_plays.to_clipboard()
